<a href="https://colab.research.google.com/github/biglalo104/Projects/blob/main/Automating%20qualitative%20data%20coding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
pip install pandas scikit-learn nltk

In [15]:
import pandas as pd

In [16]:


# Mock dataset: Customer feedback on a product
data = {
    'response_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'text': [
        "The battery life is amazing, lasts all day.",
        "I hate how heavy this device is.",
        "Screen resolution is crystal clear and beautiful.",
        "Customer service was unhelpful and rude.",
        "Battery drains too fast, needs constant charging.",
        "The build quality is cheap and feels flimsy.",
        "Love the high resolution display, great for movies.",
        "Support team took forever to reply.",
        "Very lightweight and easy to carry around.",
        "The screen is dull and colors look washed out."
    ],
    # These are our pre-defined "codes" or themes
    'theme': ['Battery', 'Design', 'Screen', 'Support', 'Battery', 'Design', 'Screen', 'Support', 'Design', 'Screen']
}

df = pd.DataFrame(data)
print("Original Data:\n", df.head(), "\n")

Original Data:
    response_id                                               text    theme
0            1        The battery life is amazing, lasts all day.  Battery
1            2                   I hate how heavy this device is.   Design
2            3  Screen resolution is crystal clear and beautiful.   Screen
3            4           Customer service was unhelpful and rude.  Support
4            5  Battery drains too fast, needs constant charging.  Battery 



In [17]:
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

#Download the necessary NLTK data files
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab') # Added to download the missing resource

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english')) # Renamed to avoid conflict and be consistent with usage

def preprocess_text(text):
  #.Lowercase the text
  text = text.lower()
  #Remoive the punctuation
  text = text.translate(str.maketrans('','',string.punctuation))
  #Tokenize(split into individual words)
  tokens = nltk.word_tokenize(text)
  #Remove stop words and tokenize
  tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
  return ' '.join(tokens) # Changed to ' '.join(tokens) for better readability of cleaned text

#Applying pre-processing to our dataset
df['clean_text'] = df['text'].apply(preprocess_text)
print("Cleaned Text Example:",df['clean_text'].iloc[0])

Cleaned Text Example: battery life amazing last day


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [18]:
#Train,validate and test model
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report,accuracy_score

#Split data into the training(80%) and testing (20%)sets
X_train,X_test,y_train,y_test = train_test_split(df['clean_text'],df['theme'],test_size=0.2,random_state=42)

#Convert text to numerical vectors using TF-IDF
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

#Train the Machine Learning Model
model = MultinomialNB()
model.fit(X_train_vec,y_train)

#Validate and Test the model
y_pred = model.predict(X_test_vec)

print("\n---Model Validation--")
print(f"Accuracy: {accuracy_score(y_test,y_pred)* 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test,y_pred))




---Model Validation--
Accuracy: 0.00%

Classification Report:
              precision    recall  f1-score   support

      Design       0.00      0.00      0.00       2.0
      Screen       0.00      0.00      0.00       0.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [19]:
#Run an Unsupervoised Learning(LDA)
from sklearn.decomposition import LatentDirichletAllocation

#Vectorize the cleaned text (ignoring extremely rare or common words)
vectorizer_unsup = TfidfVectorizer(max_df=0.9,min_df=1,stop_words='english')
dtm = vectorizer_unsup.fit_transform(df['clean_text'])

#Train the LDA model to find 3 distinct topics
lda_model = LatentDirichletAllocation(n_components=3,random_state=42)
lda_model.fit(dtm)

# Display the top words defining each discovered "Theme"
feature_names = vectorizer_unsup.get_feature_names_out()
print("\n---Discovered Themes (Unsupervised) ---")
for topic_idx, topic in enumerate(lda_model.components_):
  top_words = [feature_names[i] for i in topic.argsort()[:-6:-1]]
  print(f"Discovered Theme{topic_idx}:{','.join(top_words)}")




---Discovered Themes (Unsupervised) ---
Discovered Theme0:carry,easy,lightweight,hate,device
Discovered Theme1:battery,life,day,amazing,fast
Discovered Theme2:screen,beautiful,clear,crystal,look


In [21]:
#Implement and Analyze
#Brand new responses from a new survey
new_responses = ["The display is stunning but the battery dies in 3 hours.",
    "The support team finally fixed my issue after a week.",
    "It's so bulky, I can't take it on the go."]

# Clean the newdata using the  exact same function
new_responses_clean = [preprocess_text(r) for r in new_responses]

# Transform the text using the  vectorizer we trained earlier
new_responses_vec = vectorizer.transform(new_responses_clean)

# Predict the themes
predictions = model.predict(new_responses_vec)

print("\n---Implementing on New Data ---")
for text,theme in zip(new_responses,predictions):
  print(f"Response:'{text}'")
  print(f"Automatically Coded As:[{theme}]\n")


---Implementing on New Data ---
Response:'The display is stunning but the battery dies in 3 hours.'
Automatically Coded As:[Screen]

Response:'The support team finally fixed my issue after a week.'
Automatically Coded As:[Support]

Response:'It's so bulky, I can't take it on the go.'
Automatically Coded As:[Screen]

